# Fuzzy Factory EDA

## Project Objective

This notebook performs exploratory data analysis (EDA) on the Fuzzy Factory multi-table e-commerce dataset. The data includes website sessions, customer orders, individual order items, refunds, and product details, which together describe the full funnel from website visit to purchase and potential refund.

The main business goal of the project is to evaluate marketing performance, product sales, profitability, and refund behavior in a way that would help an e-commerce leader make decisions about channels, campaigns, and products.

The purpose of EDA in this notebook is to understand the structure and quality of each table, validate keys and relationships, and identify data issues that must be addressed before building SQL models, KPIs, and dashboards.

## Files Loaded

Planned input files for this notebook:

- `data/raw/website_sessions.csv`
- `data/raw/orders.csv`
- `data/raw/order_items.csv`
- `data/raw/order_item_refunds.csv`
- `data/raw/products.csv`
- `data/raw/website_pageviews.csv`
Note: In this notebook the working directory is the `notebooks` folder, so the code uses `../data/raw/...` to point to the `data/raw` directory in the project root.

In [1]:
import pandas as pd

df_website_sessions = pd.read_csv('../data/raw/website_sessions.csv')
df_orders = pd.read_csv('../data/raw/orders.csv')
df_order_items = pd.read_csv('../data/raw/order_items.csv')
df_order_item_refunds = pd.read_csv('../data/raw/order_item_refunds.csv')
df_products = pd.read_csv('../data/raw/products.csv')
df_website_pageviews = pd.read_csv('../data/raw/website_pageviews.csv')


## Table Overview

In this section I confirm the basic structure of each table by checking the number of rows and columns. This helps verify that the data loaded correctly and that the tables have realistic sizes given their role in the business process.

In [2]:
print("df_website_sessions shape: " + str(df_website_sessions.shape))
print("df_orders shape: " + str(df_orders.shape))
print("df_order_items shape: " + str(df_order_items.shape))
print("df_order_item_refunds shape: " + str(df_order_item_refunds.shape))
print("df_products shape: " + str(df_products.shape))
print("df_website_pageviews shape: " + str(df_website_pageviews.shape))

df_website_sessions shape: (472871, 9)
df_orders shape: (32313, 8)
df_order_items shape: (40025, 7)
df_order_item_refunds shape: (1731, 5)
df_products shape: (4, 3)
df_website_pageviews shape: (1188124, 4)


### Column Names

Here I list the column names for each table and compare them to the data dictionary. This confirms that keys, measures, and dimensions are present and correctly named before I start any cleaning or joins.

In [3]:
print("df_website_sessions columns: " + str(df_website_sessions.columns))
print("df_orders columns: " + str(df_orders.columns))
print("df_order_items columns: " + str(df_order_items.columns))
print("df_order_item_refunds columns: " + str(df_order_item_refunds.columns))
print("df_products columns: " + str(df_products.columns))
print("df_website_pageviews columns: " + str(df_website_pageviews.columns))

df_website_sessions columns: Index(['website_session_id', 'created_at', 'user_id', 'is_repeat_session',
       'utm_source', 'utm_campaign', 'utm_content', 'device_type',
       'http_referer'],
      dtype='str')
df_orders columns: Index(['order_id', 'created_at', 'website_session_id', 'user_id',
       'primary_product_id', 'items_purchased', 'price_usd', 'cogs_usd'],
      dtype='str')
df_order_items columns: Index(['order_item_id', 'created_at', 'order_id', 'product_id',
       'is_primary_item', 'price_usd', 'cogs_usd'],
      dtype='str')
df_order_item_refunds columns: Index(['order_item_refund_id', 'created_at', 'order_item_id', 'order_id',
       'refund_amount_usd'],
      dtype='str')
df_products columns: Index(['product_id', 'created_at', 'product_name'], dtype='str')
df_website_pageviews columns: Index(['website_pageview_id', 'created_at', 'website_session_id',
       'pageview_url'],
      dtype='str')


## Data Type Checks

In [4]:
print("df_website_sessions data types: " + str(df_website_sessions.dtypes))
print("df_orders data types: " + str(df_orders.dtypes))
print("df_order_items data types: " + str(df_order_items.dtypes))
print("df_order_item_refunds data types: " + str(df_order_item_refunds.dtypes))
print("df_products data types: " + str(df_products.dtypes))
print("df_website_pageviews data types: " + str(df_website_pageviews.dtypes))

df_website_sessions data types: website_session_id    int64
created_at              str
user_id               int64
is_repeat_session     int64
utm_source              str
utm_campaign            str
utm_content             str
device_type             str
http_referer            str
dtype: object
df_orders data types: order_id                int64
created_at                str
website_session_id      int64
user_id                 int64
primary_product_id      int64
items_purchased         int64
price_usd             float64
cogs_usd              float64
dtype: object
df_order_items data types: order_item_id        int64
created_at             str
order_id             int64
product_id           int64
is_primary_item      int64
price_usd          float64
cogs_usd           float64
dtype: object
df_order_item_refunds data types: order_item_refund_id      int64
created_at                  str
order_item_id             int64
order_id                  int64
refund_amount_usd       float64
dt

## Missing Value Checks

Missing values can affect attribution, funnel analysis, and revenue or refund metrics, so in this section I quantify nulls in each table before deciding how they should be handled or documented.

In [5]:
print("df_website_sessions missing values: " + str(df_website_sessions.isnull().sum()))
print("df_orders missing values: " + str(df_orders.isnull().sum()))
print("df_order_items missing values: " + str(df_order_items.isnull().sum()))
print("df_order_item_refunds missing values: " + str(df_order_item_refunds.isnull().sum()))
print("df_products missing values: " + str(df_products.isnull().sum()))
print("df_website_pageviews missing values: " + str(df_website_pageviews.isnull().sum()))

df_website_sessions missing values: website_session_id        0
created_at                0
user_id                   0
is_repeat_session         0
utm_source            83328
utm_campaign          83328
utm_content           83328
device_type               0
http_referer          39917
dtype: int64
df_orders missing values: order_id              0
created_at            0
website_session_id    0
user_id               0
primary_product_id    0
items_purchased       0
price_usd             0
cogs_usd              0
dtype: int64
df_order_items missing values: order_item_id      0
created_at         0
order_id           0
product_id         0
is_primary_item    0
price_usd          0
cogs_usd           0
dtype: int64
df_order_item_refunds missing values: order_item_refund_id    0
created_at              0
order_item_id           0
order_id                0
refund_amount_usd       0
dtype: int64
df_products missing values: product_id      0
created_at      0
product_name    0
dtype: int64
d

## Duplicate Key Checks


In [6]:
print("df_website_session duplicates: " + str(df_website_sessions.duplicated().sum()))
print("df_orders duplicates: " + str(df_orders.duplicated().sum()))
print("df_order_items duplicates: " + str(df_order_items.duplicated().sum()))
print("df_order_item_refunds duplicates: " + str(df_order_item_refunds.duplicated().sum()))
print("df_products duplicates: " + str(df_products.duplicated().sum()))
print("df_website_pageviews duplicates: " + str(df_website_pageviews.duplicated().sum()))

df_website_session duplicates: 0
df_orders duplicates: 0
df_order_items duplicates: 0
df_order_item_refunds duplicates: 0
df_products duplicates: 0
df_website_pageviews duplicates: 0


## Primary key uniqueness

For each core table, I treated the ID field as the primary key: `website_session_id`, `website_pageview_id`, `order_id`, `order_item_id`, `order_item_refund_id`, and `product_id`. Based on the missing-value checks, all of these key columns contain no null values. Duplicate-row checks also returned zero fully duplicated records across all six tables. Together, these results confirm that each ID behaves as a unique record identifier and can be safely used in joins, aggregations, and downstream KPI analysis.

## Join validation

To confirm referential integrity across the full funnel, I checked that each foreign key in child tables successfully matches a primary key in its parent table. Specifically:

- Every `orders.website_session_id` exists in `website_sessions.website_session_id`.
- Every `order_items.order_id` exists in `orders.order_id`.
- Every `order_items.product_id` exists in `products.product_id`.
- Every `order_item_refunds.order_item_id` exists in `order_items.order_item_id`.
- Every `order_item_refunds.order_id` exists in `orders.order_id`.
- Every `website_pageviews.website_session_id` exists in `website_sessions.website_session_id`.

All six checks returned **0 orphan rows**, which confirms that session, pageview, order, item, refund, and product records are consistently linked and can be safely used together in funnel, attribution, and profitability analysis.

In [7]:
checks = {
    "orders -> website_sessions": (
        ~df_orders["website_session_id"].isin(df_website_sessions["website_session_id"])
    ).sum(),
    "order_items -> orders": (
        ~df_order_items["order_id"].isin(df_orders["order_id"])
    ).sum(),
    "order_items -> products": (
        ~df_order_items["product_id"].isin(df_products["product_id"])
    ).sum(),
    "order_item_refunds -> order_items": (
        ~df_order_item_refunds["order_item_id"].isin(df_order_items["order_item_id"])
    ).sum(),
    "order_item_refunds -> orders": (
        ~df_order_item_refunds["order_id"].isin(df_orders["order_id"])
    ).sum(),
    "website_pageviews -> website_sessions": (
        ~df_website_pageviews["website_session_id"].isin(df_website_sessions["website_session_id"])
    ).sum(),
}

for check_name, orphan_count in checks.items():
    print(f"{check_name}: {orphan_count} orphan rows")

orders -> website_sessions: 0 orphan rows
order_items -> orders: 0 orphan rows
order_items -> products: 0 orphan rows
order_item_refunds -> order_items: 0 orphan rows
order_item_refunds -> orders: 0 orphan rows
website_pageviews -> website_sessions: 0 orphan rows


## Early observations

Overall, the Fuzzy Factory data looks structurally healthy based on initial EDA. Core transaction and product tables have complete keys, dates, and monetary fields, with no missing values, duplicate rows, or issues with primary key uniqueness. Website session data shows expected missingness in UTM and referer fields, reflecting untagged or direct traffic rather than broken records. Referential integrity checks confirm that orders, order items, and refunds all correctly reference their parent sessions, orders, and products, with zero orphan child rows across all tested relationships.

## Value plausibility checks

This section validates that order quantities, revenue, costs, and refund amounts are logically consistent before they are used in KPI and profitability analysis.

The checks confirm that:

- `items_purchased` contains valid positive quantities.
- `price_usd` and `cogs_usd` contain positive values.
- No order has `cogs_usd` greater than `price_usd`.
- `refund_amount_usd` contains valid positive values.
- No refund amount exceeds the price of its related order item.

These checks help confirm that financial and operational metrics are internally consistent and suitable for downstream analysis.

In [10]:
# Basic plausibility checks for orders
for col in ["items_purchased", "price_usd", "cogs_usd"]:
    s = df_orders[col]

    print(f"\nColumn: {col}")
    print(f"  count: {s.count()}, nulls: {s.isnull().sum()}")
    print(f"  negatives: {(s < 0).sum()}, zeros: {(s == 0).sum()}")
    print(
        f"  min: {s.min()}, "
        f"1%: {s.quantile(0.01)}, "
        f"99%: {s.quantile(0.99)}, "
        f"max: {s.max()}"
    )

# Confirm order-level costs do not exceed revenue
inconsistent_cogs = (df_orders["cogs_usd"] > df_orders["price_usd"]).sum()
print(f"\nRows with cogs_usd > price_usd: {inconsistent_cogs}")

# Basic plausibility checks for refunds
s = df_order_item_refunds["refund_amount_usd"]

print("\nColumn: refund_amount_usd")
print(f"  count: {s.count()}, nulls: {s.isnull().sum()}")
print(f"  negatives: {(s < 0).sum()}, zeros: {(s == 0).sum()}")
print(
    f"  min: {s.min()}, "
    f"1%: {s.quantile(0.01)}, "
    f"99%: {s.quantile(0.99)}, "
    f"max: {s.max()}"
)

print("\nRefund amount statistics:")
print(s.describe())

# Confirm refund amounts do not exceed the related item price
refund_check = df_order_item_refunds.merge(
    df_order_items[["order_item_id", "price_usd"]],
    on="order_item_id",
    how="left",
)

excessive_refunds = (
    refund_check["refund_amount_usd"] > refund_check["price_usd"]
).sum()

print(f"\nRefund amounts exceeding item price: {excessive_refunds}")


Column: items_purchased
  count: 32313, nulls: 0
  negatives: 0, zeros: 0
  min: 1, 1%: 1.0, 99%: 2.0, max: 2

Column: price_usd
  count: 32313, nulls: 0
  negatives: 0, zeros: 0
  min: 29.99, 1%: 29.99, 99%: 109.98, max: 109.98

Column: cogs_usd
  count: 32313, nulls: 0
  negatives: 0, zeros: 0
  min: 9.49, 1%: 9.49, 99%: 41.98, max: 41.98

Rows with cogs_usd > price_usd: 0

Column: refund_amount_usd
  count: 1731, nulls: 0
  negatives: 0, zeros: 0
  min: 29.99, 1%: 29.99, 99%: 59.99, max: 59.99

Refund amount statistics:
count    1731.000000
mean       49.300225
std         4.956015
min        29.990000
25%        49.990000
50%        49.990000
75%        49.990000
max        59.990000
Name: refund_amount_usd, dtype: float64

Refund amounts exceeding item price: 0


### Results

The order-level and refund-level financial fields passed all plausibility checks.

- `items_purchased` contains 32,313 complete records, with quantities ranging from 1 to 2 items per order. No negative or zero quantities were found.
- `price_usd` ranges from $29.99 to $109.98, while `cogs_usd` ranges from $9.49 to $41.98. Both fields contain no null, negative, or zero values.
- No orders have `cogs_usd` greater than `price_usd`, confirming that reported order-level costs do not exceed revenue.
- `refund_amount_usd` contains 1,731 complete records and ranges from $29.99 to $59.99. No negative or zero refund values were found.
- The average refund amount is $49.30, while the median refund amount is $49.99.
- No refund amount exceeds the original price of its related order item.

These results confirm that quantities, revenue, costs, and refund values are internally consistent and can be used reliably in downstream KPI, profitability, and refund-rate analysis.

## Open Cleaning Issues

- Convert all `created_at` fields from string to proper datetime types in each table to enable reliable time-based filtering, grouping, and trend analysis.
- Decide how to treat null UTM fields (`utm_source`, `utm_campaign`, `utm_content`) and `http_referer` in `website_sessions`, such as labeling them as untagged or direct traffic for attribution analysis.
- Run basic plausibility checks on key numeric fields (e.g., `price_usd`, `cogs_usd`, `items_purchased`, `refund_amount_usd`) to confirm there are no negative, zero, or extreme values that would distort KPIs.
- Document any assumptions made about missing marketing metadata and value ranges so downstream SQL models and dashboards can interpret the data consistently.